# S05 · Mini PRIMM "aplanar y mapear" · Jacobi 2D en una malla aplanada

> **Cómputo Paralelo · 6BM1 · ESCOM-IPN · 27/1** · Unidad V · Sesión S05 · martes 1 de diciembre de 2026 · 08:30–10:00 · Lab LS4

Antes de correr: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**. En Kaggle con P100 cambia `-arch=sm_75` por `-arch=sm_60`.
Anota la fecha y la GPU que te asignaron (celda siguiente): el hardware de Colab cambia y el protocolo exige declararlo.

Sigue el `README.md` de la práctica guiada: **escribe tu predicción antes de ejecutar** la celda de la fase Ejecutar.

In [ ]:
!nvidia-smi
!nvcc --version | tail -1
!date

## Código
Las tres celdas siguientes escriben los archivos en el disco de Colab. Si modificas el kernel (fase Modificar), edita la celda y vuelve a ejecutarla **y** la celda de compilación.

In [ ]:
%%writefile flatten_map_2d.cu
// flatten_map_2d.cu — Mini PRIMM "aplanar y mapear"
// Un paso de Jacobi 2D (promedio de los 4 vecinos) sobre una malla alto x ancho que vive
// aplanada en un arreglo 1D (por filas). El kernel 2D usa blockIdx.y/threadIdx.y; la variante
// 1D (modo "1d") tiene un TODO(1) que completa el estudiante en la fase Modificar.
//
// Cómputo Paralelo · 6BM1 · ESCOM-IPN · 27/1 · Unidad V · Sesión S05 (martes 1 de diciembre de 2026)
//
// Uso:
//   ./flatten_map_2d [alto] [ancho] [iteraciones] [bloque_x] [bloque_y] [modo 2d|1d]
//                    [copiar_cada_iter 0|1] [fila_objetivo] [col_objetivo]
// Por defecto: 1024 1024 100 16 16 2d 0 3 5
// Variable de entorno CPU_REPS (por defecto 5): repeticiones de la referencia CPU.
//
// Verificación local sin GPU:  nvcc -c -arch=sm_75 flatten_map_2d.cu   (solo compila)
// En Colab/Kaggle:             nvcc -O2 -arch=sm_75 flatten_map_2d.cu -o flatten_map_2d && ./flatten_map_2d
//
// Experimento deliberadamente incorrecto (solo en Colab, fase Investigar):
//   nvcc -O2 -arch=sm_75 -DSIN_GUARDIA flatten_map_2d.cu -o sin_guardia
//   Quita la guardia de rango del kernel 2D: con dimensiones no múltiplos del bloque los hilos
//   sobrantes escriben fuera del arreglo (comportamiento indefinido a propósito).

#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cmath>
#include <chrono>
#include <algorithm>
#include <vector>
#include <cuda_runtime.h>

// Toda llamada al API se revisa. Un error silencioso en cudaMalloc o cudaMemcpy produce
// resultados basura que "no dan error" en el kernel (idea errónea 27 del Anexo A).
#define CUDA_CHECK(call)                                                              \
    do {                                                                              \
        cudaError_t err__ = (call);                                                   \
        if (err__ != cudaSuccess) {                                                   \
            fprintf(stderr, "Error CUDA en %s:%d: %s\n", __FILE__, __LINE__,          \
                    cudaGetErrorString(err__));                                       \
            exit(EXIT_FAILURE);                                                       \
        }                                                                             \
    } while (0)

static const int   REPS      = 5;       // repeticiones medidas (protocolo del curso)
static const float TOP_VALUE = 100.0f;  // frontera superior "caliente"; el resto empieza en 0
static const float TOL       = 1e-4f;   // tolerancia absoluta de la verificación vs. CPU

// Índice lineal de (fila r, columna c) en una malla de ancho w guardada por filas.
__host__ __device__ inline int lin(int r, int c, int w) { return r * w + c; }

// ---------------------------------------------------------------------------------------
// Kernel 2D: cada hilo procesa una celda; x recorre columnas (lo contiguo), y recorre filas.
// ---------------------------------------------------------------------------------------
__global__ void jacobi_step_2d(const float *in, float *out, int height, int width,
                               int target_r, int target_c, int *who)
{
    int c = blockIdx.x * blockDim.x + threadIdx.x;   // columna
    int r = blockIdx.y * blockDim.y + threadIdx.y;   // fila
#ifndef SIN_GUARDIA
    // Guardia: cuando alto o ancho no son múltiplos del bloque sobran hilos. Sin esta línea
    // escriben fuera del arreglo.
    if (r >= height || c >= width) return;
#endif
    int idx = lin(r, c, width);

    // Registro de "quién procesó la celda objetivo" (para comparar con tu predicción).
    if (r == target_r && c == target_c) {
        who[0] = blockIdx.x;  who[1] = blockIdx.y;
        who[2] = threadIdx.x; who[3] = threadIdx.y;
        who[4] = idx;
    }

    // La frontera se copia tal cual (condición de Dirichlet fija).
    if (r == 0 || r == height - 1 || c == 0 || c == width - 1) {
        out[idx] = in[idx];
        return;
    }
    // Celda interior: promedio de arriba, abajo, izquierda y derecha (leídos de `in`, nunca de `out`).
    out[idx] = 0.25f * (in[idx - width] + in[idx + width] + in[idx - 1] + in[idx + 1]);
}

// ---------------------------------------------------------------------------------------
// Kernel 1D: cada hilo recibe un índice lineal y debe recuperar (fila, columna).
// ---------------------------------------------------------------------------------------
__global__ void jacobi_step_1d(const float *in, float *out, int height, int width,
                               int target_r, int target_c, int *who)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;   // un índice lineal por hilo
    if (idx >= height * width) return;                 // guardia del último bloque

    /* TODO(1): calcula la fila r y la columna c a partir de idx, sabiendo que idx = r*width + c.
       Mientras r y c valgan -1, la guardia de abajo descarta a todos los hilos y el programa
       imprime "TODO pendiente" en modo 1d. */
    int r = -1;
    int c = -1;
    if (r < 0 || c < 0) return;

    if (r == target_r && c == target_c) {
        who[0] = blockIdx.x;  who[1] = 0;
        who[2] = threadIdx.x; who[3] = 0;
        who[4] = idx;
    }
    if (r == 0 || r == height - 1 || c == 0 || c == width - 1) {
        out[idx] = in[idx];
        return;
    }
    out[idx] = 0.25f * (in[idx - width] + in[idx + width] + in[idx - 1] + in[idx + 1]);
}

// ---------------------------------------------------------------------------------------
// Anfitrión: inicialización, referencia secuencial, utilidades.
// ---------------------------------------------------------------------------------------
static void init_grid(float *g, int h, int w)
{
    for (int r = 0; r < h; r++)
        for (int c = 0; c < w; c++)
            g[lin(r, c, w)] = (r == 0) ? TOP_VALUE : 0.0f;
}

// Referencia secuencial (misma aritmética en float que el kernel). Deja el resultado en `out`.
static void jacobi_cpu(const float *init, float *out, float *tmp, int h, int w, int iters)
{
    float *cur = out, *nxt = tmp;
    memcpy(cur, init, (size_t)h * w * sizeof(float));
    memcpy(nxt, init, (size_t)h * w * sizeof(float));   // la frontera ya queda copiada en ambos
    for (int it = 0; it < iters; it++) {
        for (int r = 1; r < h - 1; r++)
            for (int c = 1; c < w - 1; c++) {
                int idx = lin(r, c, w);
                nxt[idx] = 0.25f * (cur[idx - w] + cur[idx + w] + cur[idx - 1] + cur[idx + 1]);
            }
        std::swap(cur, nxt);
    }
    if (cur != out) memcpy(out, cur, (size_t)h * w * sizeof(float));
}

static double now_s(void)
{
    using namespace std::chrono;
    return duration<double>(steady_clock::now().time_since_epoch()).count();
}

static double median(std::vector<double> v)
{
    std::sort(v.begin(), v.end());
    size_t n = v.size();
    return (n % 2) ? v[n / 2] : 0.5 * (v[n / 2 - 1] + v[n / 2]);
}

static double checksum(const float *g, size_t n)
{
    double s = 0.0;
    for (size_t i = 0; i < n; i++) s += g[i];
    return s;
}

static int ceil_div(int a, int b) { return (a + b - 1) / b; }

struct Config {
    int h, w, iters, bx, by;
    bool mode_2d;
    bool copy_each;
    int target_r, target_c;
};

// Una corrida completa de la "V3": H2D, `iters` pasos de Jacobi y D2H.
// Devuelve el tiempo solo de kernels y el tiempo total (con transferencias), ambos en segundos.
// Si copy_each es verdadero reproduce el error típico de porte: copiar ida y vuelta en cada iteración.
static void run_once(const Config &cfg, const float *h_init, float *h_out, float *h_scratch,
                     float *d_a, float *d_b, int *d_who, dim3 grid, dim3 block,
                     double *t_kernel_s, double *t_total_s)
{
    size_t bytes = (size_t)cfg.h * cfg.w * sizeof(float);
    cudaEvent_t e_total0, e_total1, e_k0, e_k1;
    CUDA_CHECK(cudaEventCreate(&e_total0)); CUDA_CHECK(cudaEventCreate(&e_total1));
    CUDA_CHECK(cudaEventCreate(&e_k0));     CUDA_CHECK(cudaEventCreate(&e_k1));
    float ms = 0.0f, ms_kernel = 0.0f;

    CUDA_CHECK(cudaEventRecord(e_total0));
    if (!cfg.copy_each) {
        // Porte correcto: los datos suben una vez, iteran en la GPU y bajan una vez.
        // Una sola subida: el kernel escribe TODAS las celdas de `out` (la frontera se copia dentro
        // del kernel), así que el segundo búfer no necesita inicializarse.
        CUDA_CHECK(cudaMemcpy(d_a, h_init, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaEventRecord(e_k0));
        for (int it = 0; it < cfg.iters; it++) {
            if (cfg.mode_2d)
                jacobi_step_2d<<<grid, block>>>(d_a, d_b, cfg.h, cfg.w, cfg.target_r, cfg.target_c, d_who);
            else
                jacobi_step_1d<<<grid, block>>>(d_a, d_b, cfg.h, cfg.w, cfg.target_r, cfg.target_c, d_who);
            CUDA_CHECK(cudaGetLastError());   // errores de lanzamiento (bloque inválido, etc.)
            std::swap(d_a, d_b);              // ping-pong: no hay copias, solo se intercambian punteros
        }
        CUDA_CHECK(cudaEventRecord(e_k1));
        CUDA_CHECK(cudaEventSynchronize(e_k1));   // sin esto se mide el encolado, no la ejecución
        CUDA_CHECK(cudaEventElapsedTime(&ms_kernel, e_k0, e_k1));
        CUDA_CHECK(cudaMemcpy(h_out, d_a, bytes, cudaMemcpyDeviceToHost));
    } else {
        // Porte incorrecto (a propósito): cada iteración sube la malla, lanza y baja el resultado.
        memcpy(h_scratch, h_init, bytes);
        for (int it = 0; it < cfg.iters; it++) {
            CUDA_CHECK(cudaMemcpy(d_a, h_scratch, bytes, cudaMemcpyHostToDevice));
            CUDA_CHECK(cudaEventRecord(e_k0));
            if (cfg.mode_2d)
                jacobi_step_2d<<<grid, block>>>(d_a, d_b, cfg.h, cfg.w, cfg.target_r, cfg.target_c, d_who);
            else
                jacobi_step_1d<<<grid, block>>>(d_a, d_b, cfg.h, cfg.w, cfg.target_r, cfg.target_c, d_who);
            CUDA_CHECK(cudaGetLastError());
            CUDA_CHECK(cudaEventRecord(e_k1));
            CUDA_CHECK(cudaEventSynchronize(e_k1));
            float ms_it = 0.0f;
            CUDA_CHECK(cudaEventElapsedTime(&ms_it, e_k0, e_k1));
            ms_kernel += ms_it;
            CUDA_CHECK(cudaMemcpy(h_scratch, d_b, bytes, cudaMemcpyDeviceToHost));
        }
        memcpy(h_out, h_scratch, bytes);
    }
    CUDA_CHECK(cudaEventRecord(e_total1));
    CUDA_CHECK(cudaEventSynchronize(e_total1));
    CUDA_CHECK(cudaEventElapsedTime(&ms, e_total0, e_total1));

    *t_kernel_s = ms_kernel * 1e-3;
    *t_total_s  = ms * 1e-3;
    CUDA_CHECK(cudaEventDestroy(e_total0)); CUDA_CHECK(cudaEventDestroy(e_total1));
    CUDA_CHECK(cudaEventDestroy(e_k0));     CUDA_CHECK(cudaEventDestroy(e_k1));
}

static void usage(const char *prog)
{
    fprintf(stderr,
        "Uso: %s [alto] [ancho] [iteraciones] [bloque_x] [bloque_y] [modo 2d|1d] "
        "[copiar_cada_iter 0|1] [fila_objetivo] [col_objetivo]\n"
        "Por defecto: 1024 1024 100 16 16 2d 0 3 5\n", prog);
}

int main(int argc, char **argv)
{
    Config cfg = {1024, 1024, 100, 16, 16, true, false, 3, 5};
    if (argc > 1 && (strcmp(argv[1], "-h") == 0 || strcmp(argv[1], "--help") == 0)) {
        usage(argv[0]);
        return 0;
    }
    if (argc > 1) cfg.h     = atoi(argv[1]);
    if (argc > 2) cfg.w     = atoi(argv[2]);
    if (argc > 3) cfg.iters = atoi(argv[3]);
    if (argc > 4) cfg.bx    = atoi(argv[4]);
    if (argc > 5) cfg.by    = atoi(argv[5]);
    if (argc > 6) {
        if (strcmp(argv[6], "2d") == 0)      cfg.mode_2d = true;
        else if (strcmp(argv[6], "1d") == 0) cfg.mode_2d = false;
        else { usage(argv[0]); return 1; }
    }
    if (argc > 7) cfg.copy_each = atoi(argv[7]) != 0;
    if (argc > 8) cfg.target_r  = atoi(argv[8]);
    if (argc > 9) cfg.target_c  = atoi(argv[9]);

    if (cfg.h < 3 || cfg.w < 3 || cfg.iters < 1 || cfg.bx < 1 || cfg.by < 1 || cfg.bx * cfg.by > 1024) {
        fprintf(stderr, "Parámetros inválidos: alto y ancho >= 3, iteraciones >= 1, 1 <= bloque_x*bloque_y <= 1024\n");
        return 1;
    }
    if (cfg.target_r < 0 || cfg.target_r >= cfg.h || cfg.target_c < 0 || cfg.target_c >= cfg.w) {
        fprintf(stderr, "La celda objetivo (%d,%d) está fuera de la malla %dx%d\n",
                cfg.target_r, cfg.target_c, cfg.h, cfg.w);
        return 1;
    }

    const size_t n     = (size_t)cfg.h * cfg.w;
    const size_t bytes = n * sizeof(float);
    int cpu_reps = 5;
    if (const char *env = getenv("CPU_REPS")) cpu_reps = std::max(1, atoi(env));

    printf("# flatten_map_2d · alto=%d ancho=%d iteraciones=%d bloque=%dx%d modo=%s copiar_cada_iter=%d\n",
           cfg.h, cfg.w, cfg.iters, cfg.bx, cfg.by, cfg.mode_2d ? "2d" : "1d", cfg.copy_each ? 1 : 0);
#ifdef SIN_GUARDIA
    printf("# ADVERTENCIA: compilado con -DSIN_GUARDIA; el kernel 2D escribe fuera de rango a propósito.\n");
#endif
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s (%d SMs, %d hilos máx. por bloque)\n", prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock);

    // --- Anfitrión -----------------------------------------------------------------------
    float *h_init    = (float *)malloc(bytes);
    float *h_ref     = (float *)malloc(bytes);
    float *h_tmp     = (float *)malloc(bytes);
    float *h_out     = (float *)malloc(bytes);
    float *h_scratch = (float *)malloc(bytes);
    if (!h_init || !h_ref || !h_tmp || !h_out || !h_scratch) { fprintf(stderr, "Sin memoria en el anfitrión\n"); return 1; }
    init_grid(h_init, cfg.h, cfg.w);

    // Referencia CPU: 1 calentamiento + cpu_reps repeticiones; se reporta la mediana.
    jacobi_cpu(h_init, h_ref, h_tmp, cfg.h, cfg.w, cfg.iters);
    std::vector<double> t_cpu_v;
    for (int rep = 0; rep < cpu_reps; rep++) {
        double t0 = now_s();
        jacobi_cpu(h_init, h_ref, h_tmp, cfg.h, cfg.w, cfg.iters);
        t_cpu_v.push_back(now_s() - t0);
    }
    double t_cpu = median(t_cpu_v);

    printf("indice lineal de (fila %d, col %d) con ancho %d: %d\n",
           cfg.target_r, cfg.target_c, cfg.w, lin(cfg.target_r, cfg.target_c, cfg.w));

    // --- Dispositivo ---------------------------------------------------------------------
    float *d_a = nullptr, *d_b = nullptr;
    int   *d_who = nullptr;
    CUDA_CHECK(cudaMalloc(&d_a, bytes));
    CUDA_CHECK(cudaMalloc(&d_b, bytes));
    CUDA_CHECK(cudaMalloc(&d_who, 5 * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_a, 0, bytes));
    CUDA_CHECK(cudaMemset(d_b, 0, bytes));
    CUDA_CHECK(cudaMemset(d_who, 0xFF, 5 * sizeof(int)));   // todos los bytes a 1 → cada int vale -1

    dim3 block, grid;
    long threads_launched;
    if (cfg.mode_2d) {
        block = dim3(cfg.bx, cfg.by);
        grid  = dim3(ceil_div(cfg.w, cfg.bx), ceil_div(cfg.h, cfg.by));   // x ↔ columnas, y ↔ filas
        threads_launched = (long)grid.x * grid.y * block.x * block.y;
        printf("malla 2D: bloques=%ux%u hilos/bloque=%ux%u\n", grid.x, grid.y, block.x, block.y);
    } else {
        block = dim3(cfg.bx * cfg.by);
        grid  = dim3(ceil_div((int)n, cfg.bx * cfg.by));
        threads_launched = (long)grid.x * block.x;
        printf("malla 1D: bloques=%u hilos/bloque=%u\n", grid.x, block.x);
    }
    printf("hilos lanzados=%ld celdas=%zu hilos ociosos=%ld (%.1f %%)\n",
           threads_launched, n, threads_launched - (long)n,
           100.0 * (double)(threads_launched - (long)n) / (double)threads_launched);

    // Calentamiento (no se cuenta) + REPS corridas medidas.
    double tk, tt;
    run_once(cfg, h_init, h_out, h_scratch, d_a, d_b, d_who, grid, block, &tk, &tt);
    std::vector<double> t_kernel_v, t_total_v;
    for (int rep = 0; rep < REPS; rep++) {
        run_once(cfg, h_init, h_out, h_scratch, d_a, d_b, d_who, grid, block, &tk, &tt);
        t_kernel_v.push_back(tk);
        t_total_v.push_back(tt);
    }
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaGetLastError());   // errores de ejecución (acceso ilegal, etc.)

    int who[5];
    CUDA_CHECK(cudaMemcpy(who, d_who, 5 * sizeof(int), cudaMemcpyDeviceToHost));

    const char *verdict = "FAIL";
    int exit_code = 1;
    if (who[4] == -1) {
        if (!cfg.mode_2d) {
            printf("TODO pendiente: el kernel 1D aún no recupera (fila, columna) a partir de idx. "
                   "Completa TODO(1) en jacobi_step_1d.\n");
            verdict = "TODO";
            exit_code = 0;
        } else {
            printf("ningún hilo procesó la celda objetivo: revisa la malla y la guardia\n");
        }
    } else {
        printf("hilo que la proceso: blockIdx=(%d,%d) threadIdx=(%d,%d) idx=%d\n",
               who[0], who[1], who[2], who[3], who[4]);
        // Verificación contra la CPU con tolerancia: el kernel "sin error" no basta.
        float max_diff = 0.0f;
        for (size_t i = 0; i < n; i++) max_diff = std::max(max_diff, fabsf(h_out[i] - h_ref[i]));
        bool ok = max_diff <= TOL;
        verdict = ok ? "PASS" : "FAIL";
        exit_code = ok ? 0 : 1;
        printf("verificacion vs CPU: %s (max |diff| = %.3e, tol = %.1e)\n", verdict, (double)max_diff, (double)TOL);
        printf("checksum = %.6f\n", checksum(h_out, n));
    }

    double t_kernel = median(t_kernel_v), t_total = median(t_total_v);
    printf("t_cpu    (mediana de %d, s) = %.6f\n", cpu_reps, t_cpu);
    printf("t_kernel (mediana de %d, s) = %.6f   <- solo kernels (cudaEvent)\n", REPS, t_kernel);
    printf("t_total  (mediana de %d, s) = %.6f   <- H2D + kernels + D2H\n", REPS, t_total);
    printf("speedup sin transferencias = %.2f\n", t_cpu / t_kernel);
    printf("speedup con transferencias = %.2f\n", t_cpu / t_total);
    printf("CSV,modo,bloque_x,bloque_y,alto,ancho,iteraciones,copiar_cada_iter,t_cpu_s,t_kernel_s,t_total_s,speedup_kernel,speedup_total,verificacion\n");
    printf("CSV,%s,%d,%d,%d,%d,%d,%d,%.6f,%.6f,%.6f,%.2f,%.2f,%s\n",
           cfg.mode_2d ? "2d" : "1d", cfg.bx, cfg.by, cfg.h, cfg.w, cfg.iters, cfg.copy_each ? 1 : 0,
           t_cpu, t_kernel, t_total, t_cpu / t_kernel, t_cpu / t_total, verdict);

    CUDA_CHECK(cudaFree(d_a)); CUDA_CHECK(cudaFree(d_b)); CUDA_CHECK(cudaFree(d_who));
    free(h_init); free(h_ref); free(h_tmp); free(h_out); free(h_scratch);
    return exit_code;
}


In [ ]:
%%writefile flatten_map_cpu.c
/* flatten_map_cpu.c — Referencia secuencial del mini PRIMM "aplanar y mapear".
 * Es la "V1" de este problema: el mismo paso de Jacobi 2D sobre la malla aplanada por filas,
 * con la misma aritmética en float que el kernel. Sirve para verificar la GPU y para T(1).
 *
 * Cómputo Paralelo · 6BM1 · ESCOM-IPN · 27/1 · Unidad V · Sesión S05 (martes 1 de diciembre de 2026)
 *
 * Compila: gcc -std=c11 -O2 -Wall -Wextra flatten_map_cpu.c -o flatten_map_cpu
 * Uso:     ./flatten_map_cpu [alto] [ancho] [iteraciones] [fila_objetivo] [col_objetivo]
 * Por defecto: 1024 1024 100 3 5
 */
#define _POSIX_C_SOURCE 199309L
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define REPS 5                 /* repeticiones medidas (protocolo del curso) */
static const float TOP_VALUE = 100.0f;

/* Índice lineal de (fila r, columna c) en una malla de ancho w guardada por filas. */
static int lin(int r, int c, int w) { return r * w + c; }

static double now_s(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;
}

static void init_grid(float *g, int h, int w)
{
    for (int r = 0; r < h; r++)
        for (int c = 0; c < w; c++)
            g[lin(r, c, w)] = (r == 0) ? TOP_VALUE : 0.0f;
}

/* `iters` pasos de Jacobi con dos arreglos (ping-pong). Deja el resultado en `out`. */
static void jacobi_cpu(const float *init, float *out, float *tmp, int h, int w, int iters)
{
    float *cur = out, *nxt = tmp;
    memcpy(cur, init, (size_t)h * w * sizeof(float));
    memcpy(nxt, init, (size_t)h * w * sizeof(float));   /* frontera copiada en ambos */
    for (int it = 0; it < iters; it++) {
        for (int r = 1; r < h - 1; r++)
            for (int c = 1; c < w - 1; c++) {
                int idx = lin(r, c, w);
                nxt[idx] = 0.25f * (cur[idx - w] + cur[idx + w] + cur[idx - 1] + cur[idx + 1]);
            }
        float *swap = cur; cur = nxt; nxt = swap;
    }
    if (cur != out) memcpy(out, cur, (size_t)h * w * sizeof(float));
}

static int cmp_double(const void *a, const void *b)
{
    double x = *(const double *)a, y = *(const double *)b;
    return (x > y) - (x < y);
}

static double median(double *v, int n)
{
    qsort(v, (size_t)n, sizeof(double), cmp_double);
    return (n % 2) ? v[n / 2] : 0.5 * (v[n / 2 - 1] + v[n / 2]);
}

int main(int argc, char **argv)
{
    int h = 1024, w = 1024, iters = 100, target_r = 3, target_c = 5;
    if (argc > 1) h        = atoi(argv[1]);
    if (argc > 2) w        = atoi(argv[2]);
    if (argc > 3) iters    = atoi(argv[3]);
    if (argc > 4) target_r = atoi(argv[4]);
    if (argc > 5) target_c = atoi(argv[5]);
    if (h < 3 || w < 3 || iters < 1) {
        fprintf(stderr, "Parámetros inválidos: alto y ancho >= 3, iteraciones >= 1\n");
        return 1;
    }
    if (target_r < 0 || target_r >= h || target_c < 0 || target_c >= w) {
        fprintf(stderr, "La celda objetivo (%d,%d) está fuera de la malla %dx%d\n", target_r, target_c, h, w);
        return 1;
    }

    size_t n = (size_t)h * w;
    float *init = malloc(n * sizeof(float));
    float *out  = malloc(n * sizeof(float));
    float *tmp  = malloc(n * sizeof(float));
    if (!init || !out || !tmp) { fprintf(stderr, "Sin memoria\n"); return 1; }
    init_grid(init, h, w);

    printf("# flatten_map_cpu · alto=%d ancho=%d iteraciones=%d (referencia secuencial)\n", h, w, iters);
    printf("indice lineal de (fila %d, col %d) con ancho %d: %d\n", target_r, target_c, w, lin(target_r, target_c, w));

    jacobi_cpu(init, out, tmp, h, w, iters);              /* calentamiento, no se cuenta */
    double t[REPS];
    for (int rep = 0; rep < REPS; rep++) {
        double t0 = now_s();
        jacobi_cpu(init, out, tmp, h, w, iters);
        t[rep] = now_s() - t0;
    }
    double t_med = median(t, REPS);

    double sum = 0.0;
    for (size_t i = 0; i < n; i++) sum += out[i];
    printf("checksum = %.6f\n", sum);
    printf("t_cpu (mediana de %d, s) = %.6f   min = %.6f   max = %.6f\n", REPS, t_med, t[0], t[REPS - 1]);
    printf("CSV,cpu,1,1,%d,%d,%d,0,%.6f,,,1.00,1.00,REF\n", h, w, iters, t_med);

    free(init); free(out); free(tmp);
    return 0;
}


In [ ]:
%%writefile flatten_map_numba.py
#!/usr/bin/env python3
"""flatten_map_numba.py — Espejo en Numba CUDA de flatten_map_2d.cu.

Mismo problema (paso de Jacobi 2D sobre una malla aplanada por filas), mismos kernels (2D completo,
1D con TODO(1)), misma referencia secuencial (NumPy) y misma salida. Sirve para:
  * verificar la lógica de índices sin GPU:  NUMBA_ENABLE_CUDASIM=1 python3 flatten_map_numba.py 10 8 3
  * correr en Colab/Kaggle con GPU real:     python3 flatten_map_numba.py 1024 1024 100 16 16 2d
Bajo CUDASIM los tiempos no significan nada (cada hilo es un hilo de Python); solo cuenta la verificación.

Cómputo Paralelo · 6BM1 · ESCOM-IPN · 27/1 · Unidad V · Sesión S05 (martes 1 de diciembre de 2026)

Uso: python3 flatten_map_numba.py [alto] [ancho] [iteraciones] [bloque_x] [bloque_y] [modo 2d|1d]
                                  [copiar_cada_iter 0|1] [fila_objetivo] [col_objetivo]
Por defecto: 1024 1024 100 16 16 2d 0 3 5
"""
import os
import sys
import time

import numpy as np
from numba import cuda, float32

CUDASIM = os.environ.get("NUMBA_ENABLE_CUDASIM", "0") == "1"
REPS = 1 if CUDASIM else 5          # bajo el simulador una repetición basta (no se mide nada)
TOP_VALUE = np.float32(100.0)
TOL = 1e-4


# -------------------------------------------------------------------------------------------
# Kernels (misma estructura que en flatten_map_2d.cu)
# -------------------------------------------------------------------------------------------
@cuda.jit
def jacobi_step_2d(inp, out, height, width, target_r, target_c, who):
    c = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x   # columna
    r = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y   # fila
    if r >= height or c >= width:                              # guardia: hilos sobrantes
        return
    idx = r * width + c
    if r == target_r and c == target_c:                        # registro de quién procesó la celda objetivo
        who[0] = cuda.blockIdx.x
        who[1] = cuda.blockIdx.y
        who[2] = cuda.threadIdx.x
        who[3] = cuda.threadIdx.y
        who[4] = idx
    if r == 0 or r == height - 1 or c == 0 or c == width - 1:  # frontera fija
        out[idx] = inp[idx]
        return
    out[idx] = float32(0.25) * (inp[idx - width] + inp[idx + width] + inp[idx - 1] + inp[idx + 1])


@cuda.jit
def jacobi_step_1d(inp, out, height, width, target_r, target_c, who):
    idx = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x  # un índice lineal por hilo
    if idx >= height * width:                                   # guardia del último bloque
        return
    # TODO(1): calcula la fila r y la columna c a partir de idx (idx = r*width + c).
    # Mientras r y c valgan -1, la guardia de abajo descarta a todos los hilos y el programa
    # imprime "TODO pendiente" en modo 1d.
    r = -1
    c = -1
    if r < 0 or c < 0:
        return
    if r == target_r and c == target_c:
        who[0] = cuda.blockIdx.x
        who[1] = 0
        who[2] = cuda.threadIdx.x
        who[3] = 0
        who[4] = idx
    if r == 0 or r == height - 1 or c == 0 or c == width - 1:
        out[idx] = inp[idx]
        return
    out[idx] = float32(0.25) * (inp[idx - width] + inp[idx + width] + inp[idx - 1] + inp[idx + 1])


# -------------------------------------------------------------------------------------------
# Anfitrión
# -------------------------------------------------------------------------------------------
def init_grid(h, w):
    g = np.zeros(h * w, dtype=np.float32)
    g[0:w] = TOP_VALUE                      # fila 0 completa: índices 0..w-1
    return g


def jacobi_cpu(init, h, w, iters):
    """Referencia secuencial (NumPy, float32, mismo orden de sumas que el kernel)."""
    cur = init.copy()
    nxt = init.copy()                       # frontera ya copiada en ambos
    for _ in range(iters):
        g = cur.reshape(h, w)
        n = nxt.reshape(h, w)
        n[1:-1, 1:-1] = np.float32(0.25) * (g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:])
        cur, nxt = nxt, cur
    return cur


def median(v):
    s = sorted(v)
    n = len(s)
    return s[n // 2] if n % 2 else 0.5 * (s[n // 2 - 1] + s[n // 2])


def ceil_div(a, b):
    return (a + b - 1) // b


def run_once(cfg, h_init, d_a, d_b, d_who, grid, block, kernel):
    """Una corrida completa: H2D, `iters` pasos y D2H. Regresa (t_kernel_s, t_total_s, h_out)."""
    h, w, iters = cfg["h"], cfg["w"], cfg["iters"]
    t_kernel = 0.0
    cuda.synchronize()
    t_total0 = time.perf_counter()
    if not cfg["copy_each"]:
        # Porte correcto: subir una vez, iterar en la GPU, bajar una vez.
        d_a.copy_to_device(h_init)      # una sola subida: el kernel escribe todas las celdas de `out`
        cuda.synchronize()
        t_k0 = time.perf_counter()
        for _ in range(iters):
            kernel[grid, block](d_a, d_b, h, w, cfg["target_r"], cfg["target_c"], d_who)
            d_a, d_b = d_b, d_a                 # ping-pong de punteros, sin copias
        cuda.synchronize()                      # sin esto se mide el encolado, no la ejecución
        t_kernel = time.perf_counter() - t_k0
        h_out = d_a.copy_to_host()
    else:
        # Porte incorrecto (a propósito): copiar ida y vuelta en cada iteración.
        h_scratch = h_init.copy()
        for _ in range(iters):
            d_a.copy_to_device(h_scratch)
            cuda.synchronize()
            t_k0 = time.perf_counter()
            kernel[grid, block](d_a, d_b, h, w, cfg["target_r"], cfg["target_c"], d_who)
            cuda.synchronize()
            t_kernel += time.perf_counter() - t_k0
            h_scratch = d_b.copy_to_host()
        h_out = h_scratch
    cuda.synchronize()
    t_total = time.perf_counter() - t_total0
    return t_kernel, t_total, h_out


def main(argv):
    cfg = {"h": 1024, "w": 1024, "iters": 100, "bx": 16, "by": 16, "mode_2d": True,
           "copy_each": False, "target_r": 3, "target_c": 5}
    if len(argv) > 1 and argv[1] in ("-h", "--help"):
        print(__doc__)
        return 0
    if len(argv) > 1: cfg["h"] = int(argv[1])
    if len(argv) > 2: cfg["w"] = int(argv[2])
    if len(argv) > 3: cfg["iters"] = int(argv[3])
    if len(argv) > 4: cfg["bx"] = int(argv[4])
    if len(argv) > 5: cfg["by"] = int(argv[5])
    if len(argv) > 6:
        if argv[6] not in ("2d", "1d"):
            print(__doc__, file=sys.stderr)
            return 1
        cfg["mode_2d"] = argv[6] == "2d"
    if len(argv) > 7: cfg["copy_each"] = int(argv[7]) != 0
    if len(argv) > 8: cfg["target_r"] = int(argv[8])
    if len(argv) > 9: cfg["target_c"] = int(argv[9])

    h, w, iters, bx, by = cfg["h"], cfg["w"], cfg["iters"], cfg["bx"], cfg["by"]
    if h < 3 or w < 3 or iters < 1 or bx < 1 or by < 1 or bx * by > 1024:
        print("Parámetros inválidos: alto y ancho >= 3, iteraciones >= 1, 1 <= bloque_x*bloque_y <= 1024",
              file=sys.stderr)
        return 1
    if not (0 <= cfg["target_r"] < h and 0 <= cfg["target_c"] < w):
        print(f"La celda objetivo ({cfg['target_r']},{cfg['target_c']}) está fuera de la malla {h}x{w}",
              file=sys.stderr)
        return 1

    n = h * w
    mode = "2d" if cfg["mode_2d"] else "1d"
    print(f"# flatten_map_numba · alto={h} ancho={w} iteraciones={iters} bloque={bx}x{by} modo={mode} "
          f"copiar_cada_iter={int(cfg['copy_each'])}" + ("  [CUDASIM: tiempos no representativos]" if CUDASIM else ""))
    if not CUDASIM:
        dev = cuda.get_current_device()
        print(f"GPU: {dev.name.decode() if isinstance(dev.name, bytes) else dev.name}")

    h_init = init_grid(h, w)

    # Referencia CPU: 1 calentamiento + REPS repeticiones; mediana.
    h_ref = jacobi_cpu(h_init, h, w, iters)
    t_cpu_v = []
    for _ in range(REPS):
        t0 = time.perf_counter()
        h_ref = jacobi_cpu(h_init, h, w, iters)
        t_cpu_v.append(time.perf_counter() - t0)
    t_cpu = median(t_cpu_v)

    print(f"indice lineal de (fila {cfg['target_r']}, col {cfg['target_c']}) con ancho {w}: "
          f"{cfg['target_r'] * w + cfg['target_c']}")

    d_a = cuda.to_device(np.zeros(n, dtype=np.float32))
    d_b = cuda.to_device(np.zeros(n, dtype=np.float32))
    d_who = cuda.to_device(np.full(5, -1, dtype=np.int32))

    if cfg["mode_2d"]:
        block = (bx, by)
        grid = (ceil_div(w, bx), ceil_div(h, by))          # x ↔ columnas, y ↔ filas
        kernel = jacobi_step_2d
        launched = grid[0] * grid[1] * bx * by
        print(f"malla 2D: bloques={grid[0]}x{grid[1]} hilos/bloque={bx}x{by}")
    else:
        block = (bx * by,)
        grid = (ceil_div(n, bx * by),)
        kernel = jacobi_step_1d
        launched = grid[0] * bx * by
        print(f"malla 1D: bloques={grid[0]} hilos/bloque={bx * by}")
    print(f"hilos lanzados={launched} celdas={n} hilos ociosos={launched - n} "
          f"({100.0 * (launched - n) / launched:.1f} %)")

    # Calentamiento (incluye la compilación JIT en GPU real) + REPS corridas medidas.
    run_once(cfg, h_init, d_a, d_b, d_who, grid, block, kernel)
    t_kernel_v, t_total_v, h_out = [], [], None
    for _ in range(REPS):
        tk, tt, h_out = run_once(cfg, h_init, d_a, d_b, d_who, grid, block, kernel)
        t_kernel_v.append(tk)
        t_total_v.append(tt)

    who = d_who.copy_to_host()
    verdict, exit_code = "FAIL", 1
    if who[4] == -1:
        if not cfg["mode_2d"]:
            print("TODO pendiente: el kernel 1D aún no recupera (fila, columna) a partir de idx. "
                  "Completa TODO(1) en jacobi_step_1d.")
            verdict, exit_code = "TODO", 0
        else:
            print("ningún hilo procesó la celda objetivo: revisa la malla y la guardia")
    else:
        print(f"hilo que la proceso: blockIdx=({who[0]},{who[1]}) threadIdx=({who[2]},{who[3]}) idx={who[4]}")
        max_diff = float(np.max(np.abs(h_out - h_ref)))
        ok = max_diff <= TOL
        verdict, exit_code = ("PASS", 0) if ok else ("FAIL", 1)
        print(f"verificacion vs CPU: {verdict} (max |diff| = {max_diff:.3e}, tol = {TOL:.1e})")
        print(f"checksum = {float(np.sum(h_out, dtype=np.float64)):.6f}")

    t_kernel, t_total = median(t_kernel_v), median(t_total_v)
    sk = t_cpu / t_kernel if t_kernel > 0 else float("nan")
    st = t_cpu / t_total if t_total > 0 else float("nan")
    print(f"t_cpu    (mediana de {REPS}, s) = {t_cpu:.6f}")
    print(f"t_kernel (mediana de {REPS}, s) = {t_kernel:.6f}   <- solo kernels (perf_counter + synchronize)")
    print(f"t_total  (mediana de {REPS}, s) = {t_total:.6f}   <- H2D + kernels + D2H")
    print(f"speedup sin transferencias = {sk:.2f}")
    print(f"speedup con transferencias = {st:.2f}")
    print("CSV,modo,bloque_x,bloque_y,alto,ancho,iteraciones,copiar_cada_iter,t_cpu_s,t_kernel_s,t_total_s,"
          "speedup_kernel,speedup_total,verificacion")
    print(f"CSV,numba-{mode},{bx},{by},{h},{w},{iters},{int(cfg['copy_each'])},{t_cpu:.6f},{t_kernel:.6f},"
          f"{t_total:.6f},{sk:.2f},{st:.2f},{verdict}")
    return exit_code


if __name__ == "__main__":
    sys.exit(main(sys.argv))


## Compilar

In [ ]:
!nvcc -O2 -arch=sm_75 flatten_map_2d.cu -o flatten_map_2d && echo 'flatten_map_2d listo'
!gcc -std=c11 -O2 -Wall -Wextra flatten_map_cpu.c -o flatten_map_cpu && echo 'flatten_map_cpu listo'

## Ejecutar (PRIMM · R)
Malla de 10 filas × 8 columnas, 3 iteraciones, bloques de 16×16. Compara con tu predicción: índice lineal de (fila 3, col 5) y `blockIdx`/`threadIdx` del hilo que la procesa.

In [ ]:
!CPU_REPS=1 ./flatten_map_2d 10 8 3 16 16 2d 0 3 5

## Investigar (PRIMM · I) · dimensiones que no son múltiplos del bloque
37 × 45 con bloques de 16 × 16. Antes de ejecutar: ¿cuántos bloques en x y en y? ¿cuántos hilos se lanzan? ¿cuántos sobran?

In [ ]:
!CPU_REPS=1 ./flatten_map_2d 37 45 3 16 16 2d 0 3 5

### Experimento incorrecto a propósito: sin guardia de rango
`-DSIN_GUARDIA` quita la línea `if (r >= height || c >= width) return;`. Los hilos sobrantes escriben fuera del arreglo.
Puede pasar cualquiera de estas tres cosas (es comportamiento indefinido): la verificación falla sin ningún error de CUDA,
aparece `illegal memory access`, o "sale bien" por casualidad. Ninguna de las tres es aceptable en tu V3.

In [ ]:
!nvcc -O2 -arch=sm_75 -DSIN_GUARDIA flatten_map_2d.cu -o sin_guardia && CPU_REPS=1 ./sin_guardia 37 45 3 16 16 2d 0 3 5; echo "codigo de salida: $?"

## Medir (protocolo del curso)
Cada corrida del programa ya hace 1 calentamiento + 5 repeticiones y reporta la **mediana** del tiempo solo de kernels
(`cudaEvent`) y del tiempo total con transferencias H2D/D2H. El barrido cubre: tamaño de bloque, con/sin transferencias,
y el error típico de copiar en cada iteración. Tarda 1–2 minutos en una T4.

In [ ]:
import subprocess, csv, datetime

H, W, ITERS = 1024, 1024, 100
configs = [
    # (bloque_x, bloque_y, modo, copiar_cada_iter)
    (32, 1, "2d", 0), (8, 8, "2d", 0), (16, 16, "2d", 0), (32, 8, "2d", 0),
    (32, 32, "2d", 0), (64, 16, "2d", 0),
    (16, 16, "2d", 1),          # copiar en cada iteración (porte incorrecto)
    (16, 16, "1d", 0),          # mapeo 1D: aparece como TODO hasta que completes TODO(1)
]
header, rows = None, []
for bx, by, mode, copy_each in configs:
    cmd = ["./flatten_map_2d", str(H), str(W), str(ITERS), str(bx), str(by), mode, str(copy_each), "3", "5"]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    lines = [l for l in out.splitlines() if l.startswith("CSV,")]
    if len(lines) < 2:
        print("sin CSV para", cmd, "\n", out); continue
    header = lines[0].split(",")[1:]
    rows.append(lines[1].split(",")[1:])
    print(" | ".join(f"{k}={v}" for k, v in zip(header, rows[-1]) if k in
          ("modo","bloque_x","bloque_y","copiar_cada_iter","t_kernel_s","t_total_s","speedup_kernel","speedup_total","verificacion")))


### T(1): la referencia secuencial con el protocolo completo (5 repeticiones, mediana, mín y máx)

In [ ]:
!./flatten_map_cpu 1024 1024 100

## Modificar (PRIMM · M) · mapeo 1D
Completa `TODO(1)` en `jacobi_step_1d` (celda `%%writefile flatten_map_2d.cu`), vuelve a compilar y ejecuta.
Predicción antes de correr: con bloques de 256 hilos, ¿qué `blockIdx.x` y `threadIdx.x` procesan (3,5) en la malla de ancho 8?

In [ ]:
!CPU_REPS=1 ./flatten_map_2d 10 8 3 16 16 1d 0 3 5

## Opcional · el mismo kernel en Numba CUDA (para equipos que harán la V3 en Python)
La primera corrida incluye la compilación JIT (por eso el calentamiento no se cuenta).

In [ ]:
!python3 flatten_map_numba.py 1024 1024 100 16 16 2d

## Gráfica: tiempo solo de kernels vs. tiempo con transferencias por configuración

In [ ]:
import matplotlib.pyplot as plt

ok_rows = [r for r in rows if r[-1] == "PASS"]
labels = [f"{r[0]} {r[1]}x{r[2]}" + (" copia/iter" if r[6] == "1" else "") for r in ok_rows]
t_kernel = [float(r[8]) * 1e3 for r in ok_rows]
t_total  = [float(r[9]) * 1e3 for r in ok_rows]
x = range(len(ok_rows))
plt.figure(figsize=(9, 4))
plt.bar([i - 0.2 for i in x], t_kernel, width=0.4, label="solo kernels")
plt.bar([i + 0.2 for i in x], t_total,  width=0.4, label="con transferencias H2D/D2H")
plt.xticks(list(x), labels, rotation=30, ha="right")
plt.ylabel("mediana de 5 (ms)"); plt.yscale("log")
plt.title(f"Jacobi 2D {H}x{W}, {ITERS} iteraciones")
plt.legend(); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()


## Guardar resultados (CSV)
El archivo lleva la fecha y la GPU. Súbelo a `resultados/` de tu repositorio si lo usas en el reporte.

In [ ]:
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip().replace(" ", "_")
fecha = datetime.date.today().isoformat()
nombre = f"resultados_S05_{fecha}_{gpu or 'gpu'}.csv"
with open(nombre, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["fecha", "gpu"] + header)
    for r in rows:
        w.writerow([fecha, gpu] + r)
print("guardado:", nombre)
try:
    from google.colab import files
    files.download(nombre)
except Exception:
    print("descarga manual desde el panel de archivos (no estás en Colab o el navegador bloqueó la descarga)")
